#  Anomaly/Fault Detection Pipeline


We already hand-verified real fault patterns during Stage 1 cleaning: 
is_transient_dropout, is_failure_event, is_early_invalid, high_spike_flag. 
These become our ground-truth labels for Stage 3 -- frame this as 
supervised fault classification using validated labels, not unsupervised 
anomaly detection from scratch.

In [1]:
# Cell 1: Load Stage 1's cleaned dataset

import pandas as pd
import numpy as np

df = pd.read_csv('/kaggle/input/notebooks/kritikabenjwal/battery-data-unification-soh-pipeline/battery_data/unified_battery_dataset_cleaned.csv')

print("Shape:", df.shape)
print("\nFault flag columns available:")
fault_cols = ['high_spike_flag', 'low_dropout_flag', 'is_transient_dropout', 'is_failure_event', 'is_early_invalid']
for col in fault_cols:
    print(f"  {col}: {df[col].sum()} rows flagged")

print("\nAny row with ANY fault flag:")
df['any_fault'] = df[fault_cols].any(axis=1)
print(df['any_fault'].sum(), "rows out of", len(df))

Shape: (399750, 22)

Fault flag columns available:
  high_spike_flag: 384 rows flagged
  low_dropout_flag: 80 rows flagged
  is_transient_dropout: 192 rows flagged
  is_failure_event: 4 rows flagged
  is_early_invalid: 42 rows flagged

Any row with ANY fault flag:
695 rows out of 399750


In [2]:
# Cell 2: Check is_failure_event rows specifically -- are these genuinely battery-related?

failure_rows = df[df['is_failure_event']]
print(failure_rows[['cell_id','cycle_index','discharge_capacity_ah','soh']].to_string())
print("\nCells with is_failure_event:", failure_rows['cell_id'].unique())

           cell_id  cycle_index  discharge_capacity_ah       soh
398638  NASA_B0040           46               0.425812  0.212906
398639  NASA_B0040           47               0.556990  0.278495
399359  NASA_B0050           20               0.096191  0.048096
399360  NASA_B0050           21               0.278085  0.139043

Cells with is_failure_event: ['NASA_B0040' 'NASA_B0050']


### Step 2 — Detect genuine degradation anomalies: knee-point acceleration in the fade curve

Knee point = well-documented battery aging phenomenon where degradation 
rate suddenly accelerates (fade rate increases sharply), typically 
signaling onset of a more severe degradation mechanism (e.g., lithium 
plating, electrode particle cracking). This is real physics, not a data 
artifact, and is exactly the kind of "invisible until it's not" risk your 
platform's pitch is built around.

Detect it per-cell: compute rolling fade-rate (slope over a trailing 
window), flag cycles where the rate suddenly steepens well beyond the 
cell's own historical baseline rate.

In [3]:
# Cell 3: Detect knee-point acceleration per cell, using a rolling fade-rate comparison

df_clean = df.dropna(subset=['soh']).copy()
df_clean = df_clean[~df_clean['is_restricted_soc'] & ~df_clean['is_low_confidence_cell']]
# also exclude rows we already know are pure data artifacts, so they don't masquerade as knee points
df_clean = df_clean[~df_clean['high_spike_flag'] & ~df_clean['is_transient_dropout'] & 
                     ~df_clean['low_dropout_flag'] & ~df_clean['is_early_invalid']]

def detect_knee_points(group, window=20, accel_threshold=2.5):
    g = group.sort_values('cycle_index').copy()
    g['rolling_fade_rate'] = g['soh'].diff().rolling(window, min_periods=10).mean()
    # baseline = median fade rate over the cell's first half of observed life
    midpoint = len(g) // 2
    baseline_rate = g['rolling_fade_rate'].iloc[:midpoint].median()
    g['is_knee_point'] = False
    if pd.notna(baseline_rate) and baseline_rate < 0:  # only makes sense if baseline is actually degrading
        # knee = current rate is more negative (faster decline) than accel_threshold x baseline
        g['is_knee_point'] = g['rolling_fade_rate'] < (baseline_rate * accel_threshold)
    return g

df_knee = df_clean.groupby('cell_id', group_keys=False).apply(detect_knee_points)

print("Total knee-point rows flagged:", df_knee['is_knee_point'].sum())
print("Cells with at least one knee point:", df_knee[df_knee['is_knee_point']]['cell_id'].nunique())
print("\nBy source:")
print(df_knee[df_knee['is_knee_point']].groupby('source')['cell_id'].nunique())

# spot check one flagged cell
example_cell = df_knee[df_knee['is_knee_point']]['cell_id'].iloc[0]
g_ex = df_knee[df_knee['cell_id']==example_cell].sort_values('cycle_index')
first_knee_cycle = g_ex[g_ex['is_knee_point']]['cycle_index'].iloc[0]
print(f"\nExample: {example_cell}, first knee-point at cycle {first_knee_cycle}")
print(g_ex[(g_ex['cycle_index'] >= first_knee_cycle-10) & (g_ex['cycle_index'] <= first_knee_cycle+10)][['cycle_index','soh','rolling_fade_rate','is_knee_point']].to_string())

Total knee-point rows flagged: 51175
Cells with at least one knee point: 217

By source:
source
calce      7
mit      138
nasa      13
snl       59
Name: cell_id, dtype: int64

Example: SNL_18650_LFP_15C_0-100_0.5/1C_a, first knee-point at cycle 535
     cycle_index       soh  rolling_fade_rate  is_knee_point
523          525  0.936493           0.000345          False
524          526  0.936450           0.000238          False
525          527  0.936354           0.000245          False
526          528  0.936265           0.000232          False
527          529  0.936263           0.000182          False
528          530  0.936173           0.000022          False
529          531  0.936145          -0.000038          False
530          532  0.936092          -0.000025          False
531          533  0.936023          -0.000040          False
532          534  0.935922          -0.000051          False
533          535  0.935851          -0.000054           True
534          536  

/tmp/ipykernel_16/3822115371.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_knee = df_clean.groupby('cell_id', group_keys=False).apply(detect_knee_points)


In [4]:
# Cell 4: Fix — require both a relative AND absolute minimum acceleration to count as a knee point

def detect_knee_points_v2(group, window=20, accel_threshold=2.5, min_abs_rate=0.001):
    g = group.sort_values('cycle_index').copy()
    g['rolling_fade_rate'] = g['soh'].diff().rolling(window, min_periods=10).mean()
    midpoint = len(g) // 2
    baseline_rate = g['rolling_fade_rate'].iloc[:midpoint].median()
    g['is_knee_point'] = False
    if pd.notna(baseline_rate) and baseline_rate < 0:
        relative_trigger = g['rolling_fade_rate'] < (baseline_rate * accel_threshold)
        absolute_trigger = g['rolling_fade_rate'] < -min_abs_rate  # must be a REAL decline, not noise
        g['is_knee_point'] = relative_trigger & absolute_trigger
    return g

df_knee2 = df_clean.groupby('cell_id', group_keys=False).apply(detect_knee_points_v2)

print("Total knee-point rows flagged:", df_knee2['is_knee_point'].sum())
print("Cells with at least one knee point:", df_knee2[df_knee2['is_knee_point']]['cell_id'].nunique())
print("\nBy source:")
print(df_knee2[df_knee2['is_knee_point']].groupby('source')['cell_id'].nunique())

# re-check the same example cell
g_ex2 = df_knee2[df_knee2['cell_id']=='SNL_18650_LFP_15C_0-100_0.5/1C_a'].sort_values('cycle_index')
print(f"\nDoes the previous false-positive example still trigger? Knee points found: {g_ex2['is_knee_point'].sum()}")

# find a genuinely strong example now
if df_knee2['is_knee_point'].sum() > 0:
    strong_example = df_knee2[df_knee2['is_knee_point']].groupby('cell_id')['rolling_fade_rate'].min().idxmin()
    g_strong = df_knee2[df_knee2['cell_id']==strong_example].sort_values('cycle_index')
    first_knee = g_strong[g_strong['is_knee_point']]['cycle_index'].iloc[0]
    print(f"\nStrongest example: {strong_example}, first knee at cycle {first_knee}")
    print(g_strong[(g_strong['cycle_index']>=first_knee-10)&(g_strong['cycle_index']<=first_knee+15)][['cycle_index','soh','rolling_fade_rate','is_knee_point']].to_string())

Total knee-point rows flagged: 8227
Cells with at least one knee point: 169

By source:
source
calce     7
mit      91
nasa     13
snl      58
Name: cell_id, dtype: int64

Does the previous false-positive example still trigger? Knee points found: 2

Strongest example: SNL_18650_NCA_35C_0-100_0.5/2C_b, first knee at cycle 12
        cycle_index       soh  rolling_fade_rate  is_knee_point
199118            2  0.975943                NaN          False
199119            3  0.973918                NaN          False
199121            5  0.924933                NaN          False
199122            6  0.932132                NaN          False
199123            7  0.939758                NaN          False
199124            8  0.942468                NaN          False
199125            9  0.935905                NaN          False
199126           10  0.931733                NaN          False
199127           11  0.945500                NaN          False
199128           12  0.944762     

/tmp/ipykernel_16/604215788.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_knee2 = df_clean.groupby('cell_id', group_keys=False).apply(detect_knee_points_v2)


In [5]:
# Cell 5: Final knee-point detector — exclude early life, require sustained duration

def detect_knee_points_v3(group, window=20, accel_threshold=2.5, min_abs_rate=0.001, 
                            early_life_exclude_pct=0.15, min_duration=10):
    g = group.sort_values('cycle_index').reset_index(drop=True).copy()
    g['rolling_fade_rate'] = g['soh'].diff().rolling(window, min_periods=15).mean()
    
    n = len(g)
    early_cutoff_idx = int(n * early_life_exclude_pct)
    baseline_rate = g['rolling_fade_rate'].iloc[early_cutoff_idx: n//2].median()
    
    g['is_knee_candidate'] = False
    if pd.notna(baseline_rate) and baseline_rate < 0:
        relative_trigger = g['rolling_fade_rate'] < (baseline_rate * accel_threshold)
        absolute_trigger = g['rolling_fade_rate'] < -min_abs_rate
        g.loc[early_cutoff_idx:, 'is_knee_candidate'] = (relative_trigger & absolute_trigger).iloc[early_cutoff_idx:]
    
    # require sustained duration: rolling sum of candidate flags over min_duration window
    g['is_knee_point'] = g['is_knee_candidate'].rolling(min_duration, min_periods=min_duration).sum() >= min_duration
    return g

df_knee3 = df_clean.groupby('cell_id', group_keys=False).apply(detect_knee_points_v3)

print("Total knee-point rows flagged:", df_knee3['is_knee_point'].sum())
print("Cells with at least one sustained knee point:", df_knee3[df_knee3['is_knee_point']]['cell_id'].nunique(), "/", df_knee3['cell_id'].nunique())
print("\nBy source:")
print(df_knee3[df_knee3['is_knee_point']].groupby('source')['cell_id'].nunique())

Total knee-point rows flagged: 4679
Cells with at least one sustained knee point: 94 / 242

By source:
source
calce     7
mit      75
snl      12
Name: cell_id, dtype: int64


/tmp/ipykernel_16/4284985415.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_knee3 = df_clean.groupby('cell_id', group_keys=False).apply(detect_knee_points_v3)


In [6]:
# Cell 6: Confirm NASA cell lengths vs. detection requirements

nasa_lengths = df_clean[df_clean['source']=='nasa'].groupby('cell_id').size()
print("NASA cell life lengths (post-exclusion):")
print(nasa_lengths.describe())
print("\nCells with <30 valid cycles (window=20 + min_duration=10 barely fits):", (nasa_lengths < 30).sum(), "/", len(nasa_lengths))

NASA cell life lengths (post-exclusion):
count     33.000000
mean      76.484848
std       57.042266
min        4.000000
25%       35.000000
50%       65.000000
75%      102.000000
max      197.000000
dtype: float64

Cells with <30 valid cycles (window=20 + min_duration=10 barely fits): 8 / 33


In [7]:
# Cell 7: Build the per-cell classification target: does this cell EVER show a knee point?

cell_knee_labels = df_knee3.groupby('cell_id')['is_knee_point'].any().reset_index()
cell_knee_labels.columns = ['cell_id', 'has_knee_point']

cell_info = df_knee3.groupby('cell_id').agg(
    source=('source','first'), cathode=('cathode','first'), 
    total_cycles=('cycle_index','max')
).reset_index()
cell_knee_labels = cell_knee_labels.merge(cell_info, on='cell_id')

print("Class balance:")
print(cell_knee_labels['has_knee_point'].value_counts())
print("\nBy source:")
print(cell_knee_labels.groupby(['source','has_knee_point']).size())

cell_knee_labels.to_csv('/kaggle/working/knee_point_labels.csv', index=False)
print("\nSaved knee_point_labels.csv")

Class balance:
has_knee_point
False    148
True      94
Name: count, dtype: int64

By source:
source  has_knee_point
calce   False              1
        True               7
mit     False             65
        True              75
nasa    False             33
snl     False             49
        True              12
dtype: int64

Saved knee_point_labels.csv


In [8]:
# Cell 8: Feature engineering — early-life features only (pre-knee-point window),
# so the classifier is predicting from signal available *before* a knee could occur

FEATURE_WINDOW_PCT = 0.15  # same cutoff used to exclude early life in knee detection

def extract_early_life_features(group):
    g = group.sort_values('cycle_index').reset_index(drop=True)
    n = len(g)
    cutoff = min(max(int(n * FEATURE_WINDOW_PCT), 15), n)
    early = g.iloc[:cutoff]
    fade = early['soh'].diff().dropna()

    return pd.Series({
        'cell_id': g['cell_id'].iloc[0],
        'source': g['source'].iloc[0],
        'cathode': g['cathode'].iloc[0],
        'total_cycles': n,
        'early_cycles_observed': cutoff,
        'early_soh_start': early['soh'].iloc[0],
        'early_soh_end': early['soh'].iloc[-1],
        'early_fade_rate_mean': fade.mean() if len(fade) else np.nan,
        'early_fade_rate_std': fade.std() if len(fade) else np.nan,
        'early_fade_rate_min': fade.min() if len(fade) else np.nan,
        'early_fade_noise_ratio': (fade.std() / abs(fade.mean())) if len(fade) and fade.mean() != 0 else np.nan,
    })

cell_features = df_clean.groupby('cell_id', group_keys=False).apply(extract_early_life_features).reset_index(drop=True)

model_df = cell_features.merge(cell_knee_labels[['cell_id', 'has_knee_point']], on='cell_id')

print("Feature matrix shape:", model_df.shape)
print("\nMissing values per column:")
print(model_df.isna().sum())
print("\nSample:")
print(model_df.head())

model_df.to_csv('/kaggle/working/cell_features.csv', index=False)
print("\nSaved cell_features.csv")

Feature matrix shape: (242, 12)

Missing values per column:
cell_id                    0
source                     0
cathode                   33
total_cycles               0
early_cycles_observed      0
early_soh_start            0
early_soh_end              0
early_fade_rate_mean       0
early_fade_rate_std        0
early_fade_rate_min        0
early_fade_noise_ratio     0
has_knee_point             0
dtype: int64

Sample:
  cell_id source cathode  total_cycles  early_cycles_observed  \
0  CS2_21  calce     LCO           781                    117   
1  CS2_33  calce     LCO           549                     82   
2  CS2_34  calce     LCO           545                     81   
3  CS2_35  calce     LCO           627                     94   
4  CS2_36  calce     LCO           609                     91   

   early_soh_start  early_soh_end  early_fade_rate_mean  early_fade_rate_std  \
0         0.881818       0.854545             -0.000235             0.015653   
1         0.950941 

/tmp/ipykernel_16/813332610.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cell_features = df_clean.groupby('cell_id', group_keys=False).apply(extract_early_life_features).reset_index(drop=True)


In [9]:
# Cell 9: Baseline classifier — stratified CV, encode categoricals, handle missing cathode

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

model_df['cathode'] = model_df['cathode'].fillna('unknown')

feature_cols_num = [
    'total_cycles', 'early_cycles_observed', 'early_soh_start', 'early_soh_end',
    'early_fade_rate_mean', 'early_fade_rate_std', 'early_fade_rate_min', 'early_fade_noise_ratio'
]
feature_cols_cat = ['source', 'cathode']

X = model_df[feature_cols_num + feature_cols_cat]
y = model_df['has_knee_point'].astype(int)

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), feature_cols_cat)
], remainder='passthrough')

clf = Pipeline([
    ('prep', preprocess),
    ('model', RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=5,
                                      class_weight='balanced', random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(clf, X, y, cv=cv, scoring=['roc_auc', 'accuracy', 'f1'])

print("ROC-AUC:", scores['test_roc_auc'].mean().round(3), "±", scores['test_roc_auc'].std().round(3))
print("Accuracy:", scores['test_accuracy'].mean().round(3), "±", scores['test_accuracy'].std().round(3))
print("F1:", scores['test_f1'].mean().round(3), "±", scores['test_f1'].std().round(3))

# Fit on full data once to inspect feature importances (not for evaluation)
clf.fit(X, y)
feat_names = clf.named_steps['prep'].get_feature_names_out()
importances = clf.named_steps['model'].feature_importances_
imp_df = pd.DataFrame({'feature': feat_names, 'importance': importances}).sort_values('importance', ascending=False)
print("\nFeature importances:")
print(imp_df.to_string(index=False))

ROC-AUC: 0.923 ± 0.024
Accuracy: 0.822 ± 0.034
F1: 0.768 ± 0.07

Feature importances:
                          feature  importance
          remainder__total_cycles    0.197464
 remainder__early_cycles_observed    0.167254
       remainder__early_soh_start    0.118470
         remainder__early_soh_end    0.110239
   remainder__early_fade_rate_min    0.107051
  remainder__early_fade_rate_mean    0.086028
remainder__early_fade_noise_ratio    0.057712
   remainder__early_fade_rate_std    0.043802
                 cat__source_nasa    0.021790
             cat__cathode_unknown    0.021047
                  cat__source_mit    0.017090
                  cat__source_snl    0.012190
                cat__source_calce    0.011599
                 cat__cathode_LCO    0.011515
                 cat__cathode_NCA    0.006212
                 cat__cathode_NMC    0.005433
                 cat__cathode_LFP    0.005104


In [10]:
# Cell 10: Ablation — does fade-rate signal alone (no cycle-count leakage) still predict knee point?

feature_cols_num_ablated = [
    'early_soh_start', 'early_soh_end',
    'early_fade_rate_mean', 'early_fade_rate_std', 'early_fade_rate_min', 'early_fade_noise_ratio'
]
# total_cycles and early_cycles_observed removed: total_cycles is structurally
# correlated with has_knee_point via the detector's own min_duration requirement,
# not because it reflects early-life degradation behavior

X_ablated = model_df[feature_cols_num_ablated + feature_cols_cat]

clf_ablated = Pipeline([
    ('prep', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), feature_cols_cat)
    ], remainder='passthrough')),
    ('model', RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=5,
                                      class_weight='balanced', random_state=42))
])

scores_ablated = cross_validate(clf_ablated, X_ablated, y, cv=cv, scoring=['roc_auc', 'accuracy', 'f1'])

print("Ablated (no cycle-count features):")
print("ROC-AUC:", scores_ablated['test_roc_auc'].mean().round(3), "±", scores_ablated['test_roc_auc'].std().round(3))
print("Accuracy:", scores_ablated['test_accuracy'].mean().round(3), "±", scores_ablated['test_accuracy'].std().round(3))
print("F1:", scores_ablated['test_f1'].mean().round(3), "±", scores_ablated['test_f1'].std().round(3))

print("\nFor comparison, full-feature run:")
print("ROC-AUC:", scores['test_roc_auc'].mean().round(3))

Ablated (no cycle-count features):
ROC-AUC: 0.884 ± 0.031
Accuracy: 0.806 ± 0.043
F1: 0.739 ± 0.058

For comparison, full-feature run:
ROC-AUC: 0.923


In [11]:
# Cell 11: Ablation 2 — remove 'source' (proxy for NASA's all-negative label), keep only physical features

feature_cols_cat_no_source = ['cathode']

X_no_source = model_df[feature_cols_num_ablated + feature_cols_cat_no_source]

clf_no_source = Pipeline([
    ('prep', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), feature_cols_cat_no_source)
    ], remainder='passthrough')),
    ('model', RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=5,
                                      class_weight='balanced', random_state=42))
])

scores_no_source = cross_validate(clf_no_source, X_no_source, y, cv=cv, scoring=['roc_auc', 'accuracy', 'f1'])

print("No source, no cycle-count (physical features only):")
print("ROC-AUC:", scores_no_source['test_roc_auc'].mean().round(3), "±", scores_no_source['test_roc_auc'].std().round(3))
print("Accuracy:", scores_no_source['test_accuracy'].mean().round(3), "±", scores_no_source['test_accuracy'].std().round(3))
print("F1:", scores_no_source['test_f1'].mean().round(3), "±", scores_no_source['test_f1'].std().round(3))

# Sanity check: what does the model predict for NASA cells specifically, without seeing 'source'?
clf_no_source.fit(X_no_source, y)
model_df['pred_no_source'] = clf_no_source.predict(X_no_source)
print("\nPredictions on NASA cells (source hidden from model):")
print(model_df[model_df['source']=='nasa']['pred_no_source'].value_counts())

No source, no cycle-count (physical features only):
ROC-AUC: 0.876 ± 0.033
Accuracy: 0.81 ± 0.06
F1: 0.742 ± 0.081

Predictions on NASA cells (source hidden from model):
pred_no_source
0    33
Name: count, dtype: int64


In [12]:
# Cell 12: Leave-one-source-out CV — train on 3 sources, test on the 4th, rotate

from sklearn.model_selection import GroupKFold

groups = model_df['source']
logo = GroupKFold(n_splits=model_df['source'].nunique())

results = []
for train_idx, test_idx in logo.split(X_no_source, y, groups):
    test_source = model_df['source'].iloc[test_idx].unique()[0]
    clf_no_source.fit(X_no_source.iloc[train_idx], y.iloc[train_idx])

    # skip AUC if held-out source has only one class present (e.g. NASA: all negative)
    y_test = y.iloc[test_idx]
    if y_test.nunique() < 2:
        auc = np.nan
    else:
        from sklearn.metrics import roc_auc_score
        auc = roc_auc_score(y_test, clf_no_source.predict_proba(X_no_source.iloc[test_idx])[:, 1])

    acc = clf_no_source.score(X_no_source.iloc[test_idx], y_test)
    results.append({'held_out_source': test_source, 'n_test': len(test_idx), 'auc': auc, 'accuracy': acc})

results_df = pd.DataFrame(results)
print(results_df)

  held_out_source  n_test       auc  accuracy
0             mit     140  0.309949  0.457143
1             snl      61  0.421769  0.459016
2            nasa      33       NaN  0.454545
3           calce       8  0.428571  0.125000


In [13]:
# Cell 13: Compare feature distributions across sources — look for batch effects

diagnostic_cols = ['early_soh_start', 'early_soh_end', 'early_fade_rate_mean',
                    'early_fade_rate_std', 'early_fade_rate_min', 'early_fade_noise_ratio']

print(model_df.groupby('source')[diagnostic_cols].describe().T)

source                             calce           mit        nasa        snl
early_soh_start        count    8.000000    140.000000   33.000000  61.000000
                       mean     0.907061      0.997689    0.705544   0.960109
                       std      0.030797      0.001638    0.231056   0.030709
                       min      0.881321      0.993982    0.321737   0.859728
                       25%      0.882040      0.996277    0.449029   0.942078
                       50%      0.897500      0.998441    0.833338   0.970654
                       75%      0.919553      0.998847    0.873090   0.985403
                       max      0.956204      1.001224    1.017669   1.001696
early_soh_end          count    8.000000    140.000000   33.000000  61.000000
                       mean     0.882089      0.999524    0.752121   0.885575
                       std      0.030571      0.004379    0.176412   0.047983
                       min      0.853126      0.982065    0.3912

In [14]:
# Cell 14: Per-source z-score normalization, then re-test leave-one-source-out generalization

norm_cols = ['early_soh_start', 'early_soh_end', 'early_fade_rate_mean',
             'early_fade_rate_std', 'early_fade_rate_min']
# excluding early_fade_noise_ratio: already unstable within-source (MIT max 10558),
# z-scoring a feature that's already garbage doesn't fix it

for col in norm_cols:
    model_df[f'{col}_z'] = model_df.groupby('source')[col].transform(
        lambda s: (s - s.mean()) / s.std(ddof=0) if s.std(ddof=0) > 0 else 0.0
    )

feature_cols_normalized = [f'{c}_z' for c in norm_cols]
X_normalized = model_df[feature_cols_normalized + ['cathode']]

clf_normalized = Pipeline([
    ('prep', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['cathode'])
    ], remainder='passthrough')),
    ('model', RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=5,
                                      class_weight='balanced', random_state=42))
])

results_norm = []
for train_idx, test_idx in logo.split(X_normalized, y, groups):
    test_source = model_df['source'].iloc[test_idx].unique()[0]
    clf_normalized.fit(X_normalized.iloc[train_idx], y.iloc[train_idx])
    y_test = y.iloc[test_idx]
    if y_test.nunique() < 2:
        auc = np.nan
    else:
        auc = roc_auc_score(y_test, clf_normalized.predict_proba(X_normalized.iloc[test_idx])[:, 1])
    acc = clf_normalized.score(X_normalized.iloc[test_idx], y_test)
    results_norm.append({'held_out_source': test_source, 'n_test': len(test_idx), 'auc': auc, 'accuracy': acc})

print(pd.DataFrame(results_norm))

# Quick look at NASA's early-life trajectory shape, to check the cycle_index concern
print("\nNASA soh trajectory sample (first cell, first 20 rows):")
first_nasa_cell = df_clean[df_clean['source']=='nasa']['cell_id'].iloc[0]
print(df_clean[df_clean['cell_id']==first_nasa_cell][['cycle_index','soh']].head(20).to_string())

  held_out_source  n_test       auc  accuracy
0             mit     140  0.178564  0.292857
1             snl      61  0.273810  0.442623
2            nasa      33       NaN  0.696970
3           calce       8  0.285714  0.250000

NASA soh trajectory sample (first cell, first 20 rows):
        cycle_index       soh
397000            1  0.928244
397001            2  0.923164
397002            3  0.917675
397003            4  0.917631
397004            5  0.917323
397005            6  0.917831
397006            7  0.917573
397007            8  0.912878
397008            9  0.912387
397009           10  0.912307
397010           11  0.912310
397011           12  0.907101
397012           13  0.906876
397013           14  0.906720
397014           15  0.901299
397015           16  0.901053
397016           17  0.901290
397017           18  0.901534
397018           19  0.901389
397019           20  0.923513


In [15]:
# Cell 15: Feature redesign — window by relative degradation reached, not cycle-count fraction

def extract_relative_life_features(group, target_relative_soh=0.95):
    g = group.sort_values('cycle_index').reset_index(drop=True)
    n = len(g)
    baseline = g['soh'].iloc[:min(5, n)].median()

    if pd.isna(baseline) or baseline == 0:
        return pd.Series({
            'cell_id': g['cell_id'].iloc[0], 'source': g['source'].iloc[0],
            'cathode': g['cathode'].iloc[0], 'total_cycles': n,
            'cycles_to_5pct_fade': np.nan, 'reached_5pct_fade': False,
            'window_fade_rate_mean': np.nan, 'window_fade_rate_std': np.nan,
            'window_fade_rate_min': np.nan, 'early_accel_ratio': np.nan,
        })

    relative_soh = g['soh'] / baseline
    below = relative_soh <= target_relative_soh
    reached = bool(below.any())
    idx_cross = below.idxmax() if reached else n - 1

    window = g.iloc[:idx_cross + 1]
    fade = window['soh'].diff().dropna()

    half = max(len(fade) // 2, 1)
    first_half_rate = fade.iloc[:half].mean() if len(fade) >= 2 else np.nan
    second_half_rate = fade.iloc[half:].mean() if len(fade) >= 2 else np.nan
    if pd.notna(first_half_rate) and first_half_rate != 0 and pd.notna(second_half_rate):
        accel_ratio = second_half_rate / first_half_rate
    else:
        accel_ratio = np.nan

    return pd.Series({
        'cell_id': g['cell_id'].iloc[0],
        'source': g['source'].iloc[0],
        'cathode': g['cathode'].iloc[0],
        'total_cycles': n,
        'cycles_to_5pct_fade': idx_cross + 1,
        'reached_5pct_fade': reached,
        'window_fade_rate_mean': fade.mean() if len(fade) else np.nan,
        'window_fade_rate_std': fade.std() if len(fade) else np.nan,
        'window_fade_rate_min': fade.min() if len(fade) else np.nan,
        'early_accel_ratio': accel_ratio,
    })

cell_features_v2 = df_clean.groupby('cell_id', group_keys=False).apply(extract_relative_life_features).reset_index(drop=True)

# NASA excluded inline here (confirmed noisy, not a reliable knee-point source --
# see later NASA investigation cells) rather than depending on cell_knee_labels_final,
# which is defined much later in this notebook (Cell 28) and isn't available yet
# during a fresh top-to-bottom run/Kaggle Save Version.
cell_knee_labels_no_nasa = cell_knee_labels[cell_knee_labels['source'] != 'nasa'][['cell_id', 'has_knee_point']]
model_df_v2 = cell_features_v2.merge(cell_knee_labels_no_nasa, on='cell_id')
model_df_v2['cathode'] = model_df_v2['cathode'].fillna('unknown')

print("Shape:", model_df_v2.shape)
print("\nreached_5pct_fade by source (cells that never hit the threshold at all are a separate signal worth watching):")
print(model_df_v2.groupby('source')['reached_5pct_fade'].value_counts())
print("\nMissing values:")
print(model_df_v2.isna().sum())
print("\nPer-source distribution of new features:")
print(model_df_v2.groupby('source')[['cycles_to_5pct_fade','window_fade_rate_mean','early_accel_ratio']].describe().T)

Shape: (209, 11)

reached_5pct_fade by source (cells that never hit the threshold at all are a separate signal worth watching):
source  reached_5pct_fade
calce   True                   8
mit     True                 138
        False                  2
snl     True                  61
Name: count, dtype: int64

Missing values:
cell_id                  0
source                   0
cathode                  0
total_cycles             0
cycles_to_5pct_fade      0
reached_5pct_fade        0
window_fade_rate_mean    0
window_fade_rate_std     2
window_fade_rate_min     0
early_accel_ratio        2
has_knee_point           0
dtype: int64

Per-source distribution of new features:
source                            calce          mit          snl
cycles_to_5pct_fade   count    8.000000   140.000000    61.000000
                      mean    46.125000   606.307143   435.737705
                      std     38.464966   301.999783   702.477399
                      min      3.000000    63.000000   

/tmp/ipykernel_16/346281272.py:46: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cell_features_v2 = df_clean.groupby('cell_id', group_keys=False).apply(extract_relative_life_features).reset_index(drop=True)


In [16]:
# Cell 16: Sustained-crossing threshold + slope-difference acceleration feature

from numpy.polynomial import polynomial as P

def extract_relative_life_features_v3(group, target_relative_soh=0.95, min_consecutive=3):
    g = group.sort_values('cycle_index').reset_index(drop=True)
    n = len(g)
    baseline = g['soh'].iloc[:min(5, n)].median()

    empty = pd.Series({
        'cell_id': g['cell_id'].iloc[0], 'source': g['source'].iloc[0],
        'cathode': g['cathode'].iloc[0], 'total_cycles': n,
        'cycles_to_5pct_fade': np.nan, 'reached_5pct_fade': False,
        'window_fade_rate_mean': np.nan, 'window_fade_rate_std': np.nan,
        'window_fade_rate_min': np.nan, 'accel_slope_diff': np.nan,
    })

    if pd.isna(baseline) or baseline == 0:
        return empty

    relative_soh = (g['soh'] / baseline).values
    below = relative_soh <= target_relative_soh

    # sustained crossing: first index where 'below' holds for min_consecutive in a row
    idx_cross = None
    run = 0
    for i, b in enumerate(below):
        run = run + 1 if b else 0
        if run >= min_consecutive:
            idx_cross = i - min_consecutive + 1
            break
    reached = idx_cross is not None
    if not reached:
        idx_cross = n - 1

    window = g.iloc[:idx_cross + 1]
    fade = window['soh'].diff().dropna()

    if len(fade) < 4:
        first_slope = second_slope = np.nan
    else:
        half = len(fade) // 2
        x1 = np.arange(half); y1 = fade.iloc[:half].values
        x2 = np.arange(len(fade) - half); y2 = fade.iloc[half:].values
        first_slope = np.polyfit(x1, y1, 1)[0] if half >= 2 else np.nan
        second_slope = np.polyfit(x2, y2, 1)[0] if len(x2) >= 2 else np.nan

    accel_slope_diff = (second_slope - first_slope) if pd.notna(first_slope) and pd.notna(second_slope) else np.nan

    return pd.Series({
        'cell_id': g['cell_id'].iloc[0],
        'source': g['source'].iloc[0],
        'cathode': g['cathode'].iloc[0],
        'total_cycles': n,
        'cycles_to_5pct_fade': idx_cross + 1,
        'reached_5pct_fade': reached,
        'window_fade_rate_mean': fade.mean() if len(fade) else np.nan,
        'window_fade_rate_std': fade.std() if len(fade) else np.nan,
        'window_fade_rate_min': fade.min() if len(fade) else np.nan,
        'accel_slope_diff': accel_slope_diff,
    })

cell_features_v3 = df_clean.groupby('cell_id', group_keys=False).apply(extract_relative_life_features_v3).reset_index(drop=True)

# NASA excluded inline (same reasoning as Cell 15) rather than depending on
# cell_knee_labels_final, which isn't defined yet at this point in a fresh run
cell_knee_labels_no_nasa = cell_knee_labels[cell_knee_labels['source'] != 'nasa'][['cell_id', 'has_knee_point']]
model_df_v3 = cell_features_v3.merge(cell_knee_labels_no_nasa, on='cell_id')
model_df_v3['cathode'] = model_df_v3['cathode'].fillna('unknown')

print("Shape:", model_df_v3.shape)
print("\nreached_5pct_fade by source (now requiring sustained dip, not first touch):")
print(model_df_v3.groupby('source')['reached_5pct_fade'].value_counts())
print("\ncycles_to_5pct_fade by source (compare medians to v2 — NASA should move up if v2's crossings were noise):")
print(model_df_v3.groupby('source')['cycles_to_5pct_fade'].describe())
print("\naccel_slope_diff by source (should no longer show -999/241-scale outliers):")
print(model_df_v3.groupby('source')['accel_slope_diff'].describe())
print("\nMissing values:")
print(model_df_v3.isna().sum())

Shape: (209, 11)

reached_5pct_fade by source (now requiring sustained dip, not first touch):
source  reached_5pct_fade
calce   True                   8
mit     True                 137
        False                  3
snl     True                  61
Name: count, dtype: int64

cycles_to_5pct_fade by source (compare medians to v2 — NASA should move up if v2's crossings were noise):
        count        mean         std   min    25%    50%    75%     max
source                                                                  
calce     8.0  153.375000   40.145939  87.0  138.0  161.0  182.5   196.0
mit     140.0  620.714286  299.709699  63.0  374.5  621.5  778.5  1645.0
snl      61.0  454.000000  739.233950   4.0   23.0   59.0  381.0  2627.0

accel_slope_diff by source (should no longer show -999/241-scale outliers):
        count          mean       std       min           25%           50%  \
source                                                                        
calce     8.0 -

/tmp/ipykernel_16/2510693688.py:63: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cell_features_v3 = df_clean.groupby('cell_id', group_keys=False).apply(extract_relative_life_features_v3).reset_index(drop=True)


In [17]:
# Cell 17: Relativized features — fraction-of-life timing + rate-normalized acceleration

EPS = 1e-6

model_df_v4 = model_df_v3.copy()
model_df_v4['frac_life_to_5pct_fade'] = model_df_v4['cycles_to_5pct_fade'] / model_df_v4['total_cycles']
model_df_v4['relative_accel'] = model_df_v4['accel_slope_diff'] / (model_df_v4['window_fade_rate_mean'].abs() + EPS)

print("frac_life_to_5pct_fade by source (should be far more comparable in scale than raw cycle counts):")
print(model_df_v4.groupby('source')['frac_life_to_5pct_fade'].describe())

print("\nrelative_accel by source (should be far more comparable in scale than raw accel_slope_diff):")
print(model_df_v4.groupby('source')['relative_accel'].describe())

print("\nCorrelation of new features with has_knee_point (quick sanity check, not a model):")
for col in ['frac_life_to_5pct_fade', 'relative_accel']:
    print(col, model_df_v4[col].corr(model_df_v4['has_knee_point'].astype(int)))

print("\nMissing values:")
print(model_df_v4[['frac_life_to_5pct_fade', 'relative_accel']].isna().sum())

model_df_v4.to_csv('/kaggle/working/cell_features_v4.csv', index=False)
print("\nSaved cell_features_v4.csv")

frac_life_to_5pct_fade by source (should be far more comparable in scale than raw cycle counts):
        count      mean       std       min       25%       50%       75%  \
source                                                                      
calce     8.0  0.224774  0.069950  0.116279  0.175059  0.247102  0.276623   
mit     140.0  0.738755  0.112129  0.370588  0.725414  0.763238  0.797561   
snl      61.0  0.161080  0.189345  0.006260  0.037406  0.080729  0.195050   

             max  
source            
calce   0.303030  
mit     1.000000  
snl     0.734653  

relative_accel by source (should be far more comparable in scale than raw accel_slope_diff):
        count      mean       std       min       25%       50%       75%  \
source                                                                      
calce     8.0 -0.050647  0.042499 -0.101998 -0.081753 -0.050728 -0.030743   
mit     140.0  0.000327  0.008088 -0.013770 -0.004497 -0.001691  0.002720   
snl      60.0 -0.049

In [18]:
# Cell 18: Check SoH at end-of-recording, and window-length reliability, by source

def last_soh_and_window_len(group, target_relative_soh=0.95, min_consecutive=3):
    g = group.sort_values('cycle_index').reset_index(drop=True)
    baseline = g['soh'].iloc[:min(5, len(g))].median()
    last_soh_frac = g['soh'].iloc[-1] / baseline if pd.notna(baseline) and baseline != 0 else np.nan
    return pd.Series({
        'cell_id': g['cell_id'].iloc[0],
        'source': g['source'].iloc[0],
        'last_relative_soh': last_soh_frac,   # e.g. 0.80 means recording stopped at 80% of original SoH
        'total_cycles': len(g),
    })

eol_check = df_clean.groupby('cell_id', group_keys=False).apply(last_soh_and_window_len).reset_index(drop=True)

print("Relative SoH at last recorded cycle, by source (this tells us what 'end of test' actually means per source):")
print(eol_check.groupby('source')['last_relative_soh'].describe())

# Reliability check for the acceleration feature: how many fade-points fall inside
# each cell's 5%-fade window? Short windows can't support a trustworthy slope estimate.
window_lengths = model_df_v3.merge(
    df_clean.groupby('cell_id').size().rename('n_rows_total').reset_index(), on='cell_id'
)
window_lengths['approx_window_points'] = window_lengths['cycles_to_5pct_fade']

print("\ncycles_to_5pct_fade (proxy for window length feeding the accel estimate) — how many are under 5 points, by source:")
print(model_df_v3.groupby('source')['cycles_to_5pct_fade'].apply(lambda s: (s < 5).sum()).rename('n_cells_window_under_5pts'))
print(model_df_v3.groupby('source').size().rename('n_cells_total'))

Relative SoH at last recorded cycle, by source (this tells us what 'end of test' actually means per source):
        count      mean       std       min       25%       50%       75%  \
source                                                                      
calce     8.0  0.358617  0.252447  0.056732  0.190952  0.303609  0.482678   
mit     140.0  0.815179  0.041044  0.762187  0.772940  0.819590  0.828449   
nasa     33.0  0.827283  0.166452  0.506744  0.747232  0.788031  0.893584   
snl      61.0  0.791579  0.127128  0.022070  0.778000  0.801752  0.838190   

             max  
source            
calce   0.864583  
mit     0.964005  
nasa    1.441733  
snl     0.932883  

cycles_to_5pct_fade (proxy for window length feeding the accel estimate) — how many are under 5 points, by source:
source
calce    0
mit      0
snl      1
Name: n_cells_window_under_5pts, dtype: int64
source
calce      8
mit      140
snl       61
Name: n_cells_total, dtype: int64


/tmp/ipykernel_16/3443058690.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  eol_check = df_clean.groupby('cell_id', group_keys=False).apply(last_soh_and_window_len).reset_index(drop=True)


In [19]:
# Cell 19: Redefine life-fraction denominator as cycles-to-80%-SoH (fixed physical 
# milestone, not "whatever cycle recording happened to stop at"), and smooth SoH 
# before computing fade rates/slopes to reduce per-cycle noise sensitivity.

SMOOTH_WINDOW = 3  # rolling median, small enough not to blur real knee behavior

def extract_relative_life_features_v4(group, target_relative_soh=0.95, eol_relative_soh=0.80, min_consecutive=3):
    g = group.sort_values('cycle_index').reset_index(drop=True)
    n = len(g)
    baseline = g['soh'].iloc[:min(5, n)].median()

    empty = pd.Series({
        'cell_id': g['cell_id'].iloc[0], 'source': g['source'].iloc[0],
        'cathode': g['cathode'].iloc[0], 'total_cycles': n,
        'cycles_to_5pct_fade': np.nan, 'reached_5pct_fade': False,
        'cycles_to_80pct_soh': np.nan, 'reached_80pct_soh': False,
        'frac_life_to_5pct_fade_v2': np.nan,
        'window_fade_rate_mean_smooth': np.nan, 'accel_slope_diff_smooth': np.nan,
    })
    if pd.isna(baseline) or baseline == 0:
        return empty

    soh_smooth = g['soh'].rolling(SMOOTH_WINDOW, min_periods=1, center=True).median()
    relative_soh = (soh_smooth / baseline).values

    def sustained_crossing(rel, thresh):
        below = rel <= thresh
        run = 0
        for i, b in enumerate(below):
            run = run + 1 if b else 0
            if run >= min_consecutive:
                return i - min_consecutive + 1
        return None

    idx_5pct = sustained_crossing(relative_soh, target_relative_soh)
    idx_eol = sustained_crossing(relative_soh, eol_relative_soh)
    reached_5pct = idx_5pct is not None
    reached_eol = idx_eol is not None
    idx_5pct_final = idx_5pct if reached_5pct else n - 1
    idx_eol_final = idx_eol if reached_eol else n - 1

    window = g.iloc[:idx_5pct_final + 1]
    fade_smooth = soh_smooth.iloc[:idx_5pct_final + 1].diff().dropna()

    if len(fade_smooth) < 4:
        first_slope = second_slope = np.nan
    else:
        half = len(fade_smooth) // 2
        x1 = np.arange(half); y1 = fade_smooth.iloc[:half].values
        x2 = np.arange(len(fade_smooth) - half); y2 = fade_smooth.iloc[half:].values
        first_slope = np.polyfit(x1, y1, 1)[0] if half >= 2 else np.nan
        second_slope = np.polyfit(x2, y2, 1)[0] if len(x2) >= 2 else np.nan
    accel_slope_diff_smooth = (second_slope - first_slope) if pd.notna(first_slope) and pd.notna(second_slope) else np.nan

    # Only meaningful if the cell actually reached the 80% EOL milestone; otherwise
    # we don't have a fair denominator for this cell and should say so, not guess.
    frac_v2 = (idx_5pct_final + 1) / (idx_eol_final + 1) if reached_eol and reached_5pct else np.nan

    return pd.Series({
        'cell_id': g['cell_id'].iloc[0],
        'source': g['source'].iloc[0],
        'cathode': g['cathode'].iloc[0],
        'total_cycles': n,
        'cycles_to_5pct_fade': idx_5pct_final + 1,
        'reached_5pct_fade': reached_5pct,
        'cycles_to_80pct_soh': idx_eol_final + 1,
        'reached_80pct_soh': reached_eol,
        'frac_life_to_5pct_fade_v2': frac_v2,
        'window_fade_rate_mean_smooth': fade_smooth.mean() if len(fade_smooth) else np.nan,
        'accel_slope_diff_smooth': accel_slope_diff_smooth,
    })

cell_features_v5 = df_clean.groupby('cell_id', group_keys=False).apply(extract_relative_life_features_v4).reset_index(drop=True)

# NASA excluded inline (same fix as Cells 15/16) — cell_knee_labels still includes NASA
cell_knee_labels_no_nasa = cell_knee_labels[cell_knee_labels['source'] != 'nasa'][['cell_id', 'has_knee_point']]
model_df_v5 = cell_features_v5.merge(cell_knee_labels_no_nasa, on='cell_id')
model_df_v5['cathode'] = model_df_v5['cathode'].fillna('unknown')

print("reached_80pct_soh by source (does everyone actually hit the EOL milestone, or is it CALCE-only-goes-deeper story confirmed?):")
print(model_df_v5.groupby('source')['reached_80pct_soh'].value_counts())

print("\nfrac_life_to_5pct_fade_v2 by source (5%-fade timing as a fraction of cycles-to-80%-EOL — should converge better than v1):")
print(model_df_v5.groupby('source')['frac_life_to_5pct_fade_v2'].describe())

print("\naccel_slope_diff_smooth by source (smoothed slopes — check whether NASA's spread shrank vs. the unsmoothed version):")
print(model_df_v5.groupby('source')['accel_slope_diff_smooth'].describe())

print("\nMissing values:")
print(model_df_v5.isna().sum())

reached_80pct_soh by source (does everyone actually hit the EOL milestone, or is it CALCE-only-goes-deeper story confirmed?):
source  reached_80pct_soh
calce   True                  8
mit     False                97
        True                 43
snl     True                 39
        False                22
Name: count, dtype: int64

frac_life_to_5pct_fade_v2 by source (5%-fade timing as a fraction of cycles-to-80%-EOL — should converge better than v1):
        count      mean       std       min       25%       50%       75%  \
source                                                                      
calce     8.0  0.360056  0.138998  0.119904  0.283743  0.383160  0.463942   
mit      43.0  0.698491  0.113594  0.382576  0.657192  0.743590  0.774195   
snl      39.0  0.138278  0.088304  0.013477  0.046333  0.149644  0.207366   

             max  
source            
calce   0.530726  
mit     0.812785  
snl     0.296703  

accel_slope_diff_smooth by source (smoothed slopes — chec

/tmp/ipykernel_16/3523621200.py:73: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cell_features_v5 = df_clean.groupby('cell_id', group_keys=False).apply(extract_relative_life_features_v4).reset_index(drop=True)


In [20]:
# Cell 20: Leave-one-source-out CV on the EOL-window features that actually have full coverage
# (dropping frac_life_to_5pct_fade_v2 -- non-convergent AND 56% missing, per Step 15 diagnosis)

from sklearn.model_selection import LeaveOneGroupOut, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

model_df_v5['cathode'] = model_df_v5['cathode'].fillna('unknown')

# accel_slope_diff_smooth has 2 missing values -- impute with median rather than drop,
# since dropping loses 2 whole cells for a feature we can reasonably impute
feature_cols_num = [
    'total_cycles', 'cycles_to_5pct_fade', 'window_fade_rate_mean_smooth', 'accel_slope_diff_smooth'
]
feature_cols_cat = ['cathode']  # deliberately excluding 'source' -- testing whether
                                 # degradation-shape features generalize without lab identity as a crutch

X = model_df_v5[feature_cols_num + feature_cols_cat]
y = model_df_v5['has_knee_point'].astype(int)
groups = model_df_v5['source']

preprocess = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), feature_cols_num),
    ('cat', OneHotEncoder(handle_unknown='ignore'), feature_cols_cat),
])

clf = Pipeline([
    ('prep', preprocess),
    ('model', RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=5,
                                      class_weight='balanced', random_state=42))
])

logo = LeaveOneGroupOut()

# cross_validate's aggregate score hides per-source performance -- loop manually so a
# strong source can't mask a source where the model is actually failing
from sklearn.metrics import roc_auc_score
results = []
for train_idx, test_idx in logo.split(X, y, groups):
    test_source = groups.iloc[test_idx].iloc[0]
    if y.iloc[test_idx].nunique() < 2:
        results.append({'held_out_source': test_source, 'auc': np.nan, 'n_test': len(test_idx), 'note': 'single class in test fold'})
        continue
    clf.fit(X.iloc[train_idx], y.iloc[train_idx])
    proba = clf.predict_proba(X.iloc[test_idx])[:, 1]
    auc = roc_auc_score(y.iloc[test_idx], proba)
    results.append({'held_out_source': test_source, 'auc': auc, 'n_test': len(test_idx), 'note': ''})

loso_results = pd.DataFrame(results)
print("Leave-one-source-out AUC (source excluded from features):")
print(loso_results.to_string(index=False))
print("\nMean AUC across sources with a valid test fold:", loso_results['auc'].mean().round(3))

Leave-one-source-out AUC (source excluded from features):
held_out_source      auc  n_test note
          calce 1.000000       8     
            mit 0.713026     140     
            snl 0.589286      61     

Mean AUC across sources with a valid test fold: 0.767


In [21]:
# Cell 21: Check whether 'cathode' is acting as a disguised source-identity feature,
# and inspect what the held-out MIT/SNL models actually relied on

print("Cathode x source crosstab (is cathode just a proxy for which lab the cell came from?):")
print(pd.crosstab(model_df_v5['source'], model_df_v5['cathode']))

# Refit on the two folds that produced a real (non-degenerate) score, and pull feature
# importances directly -- this tells us whether window_fade_rate_mean_smooth /
# accel_slope_diff_smooth are doing the work, or whether cathode's one-hot columns
# are dominating (the leakage signature we're checking for)
for held_out in ['mit', 'snl']:
    train_idx = model_df_v5['source'] != held_out
    clf.fit(X[train_idx], y[train_idx])
    feat_names = clf.named_steps['prep'].get_feature_names_out()
    importances = clf.named_steps['model'].feature_importances_
    imp_df = pd.DataFrame({'feature': feat_names, 'importance': importances}).sort_values('importance', ascending=False)
    print(f"\nFeature importances -- trained on all-but-{held_out}, tested on {held_out}:")
    print(imp_df.to_string(index=False))

Cathode x source crosstab (is cathode just a proxy for which lab the cell came from?):
cathode  LCO  LFP  NCA  NMC
source                     
calce      8    0    0    0
mit        0  140    0    0
snl        0   21   18   22

Feature importances -- trained on all-but-mit, tested on mit:
                          feature  importance
num__window_fade_rate_mean_smooth    0.226706
                num__total_cycles    0.226506
         num__cycles_to_5pct_fade    0.194101
     num__accel_slope_diff_smooth    0.158577
                 cat__cathode_LCO    0.061097
                 cat__cathode_NMC    0.055517
                 cat__cathode_LFP    0.054374
                 cat__cathode_NCA    0.023122

Feature importances -- trained on all-but-snl, tested on snl:
                          feature  importance
                num__total_cycles    0.393101
         num__cycles_to_5pct_fade    0.255849
num__window_fade_rate_mean_smooth    0.229984
     num__accel_slope_diff_smooth    0.112966
   

In [22]:
# Cell 22 (fix): describe() uses '50%' for the median, not 'median'

print("Correlation: total_cycles vs has_knee_point (point-biserial via groupby):")
print(model_df_v5.groupby('has_knee_point')['total_cycles'].describe()[['count','mean','50%','std']])
# If cells with has_knee_point=True are simply the longer-running cells, that's the
# leakage signature -- not a coincidence, a structural artifact of how detection works
# (Step 3d/3e: min_duration=10 sustained cycles requires enough cycles to even be eligible)

# Re-run leave-one-source-out CV WITHOUT total_cycles, to see how much of the AUC was
# actually resting on test duration rather than early-life degradation shape
feature_cols_num_v2 = [
    'cycles_to_5pct_fade', 'window_fade_rate_mean_smooth', 'accel_slope_diff_smooth'
]
X_v2 = model_df_v5[feature_cols_num_v2 + feature_cols_cat]

preprocess_v2 = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), feature_cols_num_v2),
    ('cat', OneHotEncoder(handle_unknown='ignore'), feature_cols_cat),
])
clf_v2 = Pipeline([
    ('prep', preprocess_v2),
    ('model', RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=5,
                                      class_weight='balanced', random_state=42))
])

results_v2 = []
for train_idx, test_idx in logo.split(X_v2, y, groups):
    test_source = groups.iloc[test_idx].iloc[0]
    if y.iloc[test_idx].nunique() < 2:
        results_v2.append({'held_out_source': test_source, 'auc': np.nan, 'n_test': len(test_idx), 'note': 'single class in test fold'})
        continue
    clf_v2.fit(X_v2.iloc[train_idx], y.iloc[train_idx])
    proba = clf_v2.predict_proba(X_v2.iloc[test_idx])[:, 1]
    auc = roc_auc_score(y.iloc[test_idx], proba)
    results_v2.append({'held_out_source': test_source, 'auc': auc, 'n_test': len(test_idx), 'note': ''})

print("\nLeave-one-source-out AUC WITHOUT total_cycles:")
print(pd.DataFrame(results_v2).to_string(index=False))

Correlation: total_cycles vs has_knee_point (point-biserial via groupby):
                count         mean    50%          std
has_knee_point                                        
False           115.0  1388.939130  915.0  1114.083529
True             94.0   675.553191  603.0   321.330420

Leave-one-source-out AUC WITHOUT total_cycles:
held_out_source      auc  n_test note
          calce 0.142857       8     
            mit 0.765846     140     
            snl 0.574830      61     


In [23]:
# Cell 23: Lock in total_cycles removal (it was future-information leakage, not a
# legitimate early-life predictor), and produce the honest final LOSO summary --
# excluding CALCE (n=8, unstable both directions) and NASA (no positive class)
# from the headline metric, but still reporting them separately for transparency.

final_feature_cols_num = ['cycles_to_5pct_fade', 'window_fade_rate_mean_smooth', 'accel_slope_diff_smooth']
final_feature_cols_cat = ['cathode']

X_final = model_df_v5[final_feature_cols_num + final_feature_cols_cat]

loso_summary = pd.DataFrame(results_v2)
loso_summary['reliable_estimate'] = ~loso_summary['held_out_source'].isin(['calce', 'nasa'])

print("Full leave-one-source-out results:")
print(loso_summary.to_string(index=False))

reliable = loso_summary[loso_summary['reliable_estimate'] & loso_summary['auc'].notna()]
print(f"\nHeadline cross-source AUC (MIT + SNL only, n={reliable['n_test'].sum()} cells): "
      f"{reliable['auc'].mean():.3f}")
print("CALCE (n=8) and NASA (no True labels) excluded from headline -- reported above for transparency, not averaged in.")

Full leave-one-source-out results:
held_out_source      auc  n_test note  reliable_estimate
          calce 0.142857       8                   False
            mit 0.765846     140                    True
            snl 0.574830      61                    True

Headline cross-source AUC (MIT + SNL only, n=201 cells): 0.670
CALCE (n=8) and NASA (no True labels) excluded from headline -- reported above for transparency, not averaged in.


In [24]:
# Cell 24: Lock in the Stage 3 generalization methodology and result as a permanent record

mit_with_leak = loso_results.loc[loso_results['held_out_source'] == 'mit', 'auc'].iloc[0]
mit_no_leak = loso_summary.loc[loso_summary['held_out_source'] == 'mit', 'auc'].iloc[0]
snl_no_leak = loso_summary.loc[loso_summary['held_out_source'] == 'snl', 'auc'].iloc[0]
calce_no_leak = loso_summary.loc[loso_summary['held_out_source'] == 'calce', 'auc'].iloc[0]
headline_auc = reliable['auc'].mean()
headline_n = reliable['n_test'].sum()

stage3_summary = f"""
### Stage 3 — Cross-source generalization: final approach and honest result

**What failed, and why:**
- Raw cycle-count-fraction features (v1): early_fade_noise_ratio unstable by
  construction (std/mean near zero); cycle-index windows meant different
  physical things per protocol (MIT flat at ~99.8% SoH in its "early" window,
  NASA already noisy/degraded in the same window). LOSO AUC: worse than chance.
- Z-score normalization within source: did not fix it, and made CALCE (n=8)
  results untrustworthy due to noisy in-group mean/std estimates.
- EOL-based windowing (frac_life_to_5pct_fade_v2): still didn't converge
  across sources (SNL median 0.15 vs MIT median 0.70), AND was missing for
  119/209 cells (57%) because most MIT cells never reach 80% SoH in the
  recorded window (censored data, not a bug). Dropped entirely.
- total_cycles as a feature: initially looked load-bearing (top-2 importance
  in both MIT and SNL folds, MIT AUC {mit_with_leak:.3f}), but is
  future-information leakage -- it's only known once a cell's full life is
  recorded, which isn't available at real prediction time on an in-service
  battery. Removing it changed MIT AUC {mit_with_leak:.3f} -> {mit_no_leak:.3f},
  confirming the model had been exploiting it rather than learning
  degradation shape.

**What worked:**
Window features defined by relative degradation reached (cycles to a
sustained 5% SoH drop from each cell's own early baseline), with the fade
trajectory smoothed (rolling median) before computing slope/acceleration.
No total_cycles, no source as a feature -- only cycles_to_5pct_fade,
window_fade_rate_mean_smooth, accel_slope_diff_smooth, and cathode.

**Final leave-one-source-out result:**
- MIT held out: AUC {mit_no_leak:.3f} (n=140)
- SNL held out: AUC {snl_no_leak:.3f} (n=61)
- Headline: {headline_auc:.3f} average, n={headline_n} reliable cells

**Explicitly out of scope, stated rather than hidden:**
- CALCE (n=8, single-holdout AUC {calce_no_leak:.3f}): sample too small for a
  stable LOSO estimate in either direction; not included in the headline
  number. See Cell 25a for a pooled leave-one-cell-out re-estimate.
- NASA (n=33): zero cells with has_knee_point=True in ground truth (Step 7),
  so cross-source AUC is undefined for this source -- not a model failure,
  a label-availability gap.
"""

print(stage3_summary)

with open('/kaggle/working/stage3_methodology_notes.md', 'w') as f:
    f.write(stage3_summary)
print("Saved stage3_methodology_notes.md")


### Stage 3 — Cross-source generalization: final approach and honest result

**What failed, and why:**
- Raw cycle-count-fraction features (v1): early_fade_noise_ratio unstable by
  construction (std/mean near zero); cycle-index windows meant different
  physical things per protocol (MIT flat at ~99.8% SoH in its "early" window,
  NASA already noisy/degraded in the same window). LOSO AUC: worse than chance.
- Z-score normalization within source: did not fix it, and made CALCE (n=8)
  results untrustworthy due to noisy in-group mean/std estimates.
- EOL-based windowing (frac_life_to_5pct_fade_v2): still didn't converge
  across sources (SNL median 0.15 vs MIT median 0.70), AND was missing for
  119/209 cells (57%) because most MIT cells never reach 80% SoH in the
  recorded window (censored data, not a bug). Dropped entirely.
- total_cycles as a feature: initially looked load-bearing (top-2 importance
  in both MIT and SNL folds, MIT AUC 0.713), but is
  future-information leakage -- i

In [25]:
# Cell 25a: CALCE fix -- leave-one-cell-out with pooled predictions (real improvement)

calce_idx = model_df_v5[model_df_v5['source'] == 'calce'].index
pooled_true, pooled_pred = [], []

for held_cell_idx in calce_idx:
    train_idx = model_df_v5.index.difference([held_cell_idx])
    clf_v2.fit(X_v2.loc[train_idx], y.loc[train_idx])
    proba = clf_v2.predict_proba(X_v2.loc[[held_cell_idx]])[:, 1]
    pooled_true.append(y.loc[held_cell_idx])
    pooled_pred.append(proba[0])

print("CALCE pooled leave-one-cell-out predictions:")
print(pd.DataFrame({'true': pooled_true, 'predicted_proba': pooled_pred}))
print("\nPooled AUC (8 out-of-fold predictions, not 1 holdout split):",
      roc_auc_score(pooled_true, pooled_pred).round(3) if len(set(pooled_true)) > 1 else "undefined -- still only 1 True label in CALCE, check class balance")

CALCE pooled leave-one-cell-out predictions:
   true  predicted_proba
0     1         0.717788
1     1         0.655902
2     1         0.702693
3     1         0.673963
4     1         0.512378
5     1         0.690456
6     1         0.709174
7     0         0.802335

Pooled AUC (8 out-of-fold predictions, not 1 holdout split): 0.0


In [26]:
# Cell 25c: Inspect the single CALCE False cell against the seven True cells directly --
# is this a real near-miss, or is the model's 0.817 prediction defensible given its features?

calce_detail = model_df_v5.loc[calce_idx, ['cell_id', 'has_knee_point', 'cycles_to_5pct_fade',
                                             'window_fade_rate_mean_smooth', 'accel_slope_diff_smooth']].copy()
calce_detail['predicted_proba'] = pooled_pred
print(calce_detail.to_string(index=False))

cell_id  has_knee_point  cycles_to_5pct_fade  window_fade_rate_mean_smooth  accel_slope_diff_smooth  predicted_proba
 CS2_21            True                  172                     -0.000292                 0.000005         0.717788
 CS2_33            True                  102                     -0.000460                -0.000032         0.655902
 CS2_34            True                  150                     -0.000325                -0.000014         0.702693
 CS2_35            True                  190                     -0.000195                -0.000015         0.673963
 CS2_36            True                   87                     -0.000549                -0.000004         0.512378
 CS2_37            True                  180                     -0.000244                -0.000010         0.690456
 CS2_38            True                  196                     -0.000243                -0.000018         0.709174
  CS2_8           False                  150                    

In [27]:
# Cell 25d: Update the Stage 3 methodology notes with the corrected CALCE finding
# (supersedes Cell 24's CALCE mention -- from "n=8, AUC 0.214, unstable" to
# "n=8, not separable on these features, AUC undefined/untrustworthy in either direction")

calce_addendum = """
### Addendum -- CALCE generalization: confirmed untestable, not just small

Leave-one-cell-out pooling (Cell 25a) was tried as a fix for CALCE's n=8
instability. Result: AUC 0.0, driven entirely by the single False cell
(CS2_8) ranking above all seven True cells. Direct feature inspection
(Cell 25c) shows CS2_8 is NOT separable from the True cells on any of the
three model features -- its values sit inside the True cells' range on
cycles_to_5pct_fade, window_fade_rate_mean_smooth, and
accel_slope_diff_smooth alike. This means CALCE's AUC is not just noisy
due to small n -- it is undefined in a stronger sense: the one negative
example is indistinguishable from the positives on every feature this
model has access to. Conclusion unchanged from Cell 23: exclude CALCE from
the headline cross-source metric, but state the reason precisely --
untestable due to lack of feature-level separability at this sample size,
not simply "small sample."
"""

print(calce_addendum)

with open('/kaggle/working/stage3_methodology_notes.md', 'a') as f:
    f.write(calce_addendum)
print("Appended to stage3_methodology_notes.md")


### Addendum -- CALCE generalization: confirmed untestable, not just small

Leave-one-cell-out pooling (Cell 25a) was tried as a fix for CALCE's n=8
instability. Result: AUC 0.0, driven entirely by the single False cell
(CS2_8) ranking above all seven True cells. Direct feature inspection
(Cell 25c) shows CS2_8 is NOT separable from the True cells on any of the
three model features -- its values sit inside the True cells' range on
cycles_to_5pct_fade, window_fade_rate_mean_smooth, and
accel_slope_diff_smooth alike. This means CALCE's AUC is not just noisy
due to small n -- it is undefined in a stronger sense: the one negative
example is indistinguishable from the positives on every feature this
model has access to. Conclusion unchanged from Cell 23: exclude CALCE from
the headline cross-source metric, but state the reason precisely --
untestable due to lack of feature-level separability at this sample size,
not simply "small sample."

Appended to stage3_methodology_notes.md


In [28]:
# Cell 26: NASA sensitivity check -- does relaxing min_duration reveal any real knee points,
# or does NASA genuinely have none regardless of threshold?

nasa_cells = df_clean[df_clean['source'] == 'nasa']

sensitivity_results = []
for min_dur in [10, 7, 5, 3]:
    df_nasa_test = nasa_cells.groupby('cell_id', group_keys=False).apply(
        lambda g: detect_knee_points_v3(g, min_duration=min_dur)
    )
    n_cells_flagged = df_nasa_test[df_nasa_test['is_knee_point']]['cell_id'].nunique()
    n_rows_flagged = df_nasa_test['is_knee_point'].sum()
    sensitivity_results.append({
        'min_duration': min_dur,
        'nasa_cells_with_knee': n_cells_flagged,
        'nasa_rows_flagged': n_rows_flagged,
        'total_nasa_cells': df_nasa_test['cell_id'].nunique()
    })

print("NASA knee-point detection under relaxed min_duration:")
print(pd.DataFrame(sensitivity_results).to_string(index=False))

# Also check the raw candidate flags (before the duration requirement) -- if even
# is_knee_candidate is almost never True for NASA, the problem isn't duration at all,
# it's that the accel_threshold/min_abs_rate triggers rarely/never fire for NASA's
# noisier baseline_rate estimate
df_nasa_candidates = nasa_cells.groupby('cell_id', group_keys=False).apply(
    lambda g: detect_knee_points_v3(g, min_duration=10)
)
print("\nNASA candidate rate (is_knee_candidate=True, before duration requirement), by cell:")
print(df_nasa_candidates.groupby('cell_id')['is_knee_candidate'].mean().sort_values(ascending=False).head(10))
print("\nCells where candidate rate is exactly 0 (never even momentarily crosses threshold):",
      (df_nasa_candidates.groupby('cell_id')['is_knee_candidate'].mean() == 0).sum(), "/",
      df_nasa_candidates['cell_id'].nunique())

/tmp/ipykernel_16/191337334.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_nasa_test = nasa_cells.groupby('cell_id', group_keys=False).apply(
/tmp/ipykernel_16/191337334.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_nasa_test = nasa_cells.groupby('cell_id', group_keys=False).apply(
/tmp/ipykernel_16/191337334.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior 

NASA knee-point detection under relaxed min_duration:
 min_duration  nasa_cells_with_knee  nasa_rows_flagged  total_nasa_cells
           10                     0                  0                33
            7                     2                  3                33
            5                     4                 13                33
            3                     5                 33                33

NASA candidate rate (is_knee_candidate=True, before duration requirement), by cell:
cell_id
NASA_B0034    0.214286
NASA_B0056    0.127451
NASA_B0042    0.123077
NASA_B0036    0.096447
NASA_B0045    0.085714
NASA_B0033    0.064171
NASA_B0055    0.029412
NASA_B0054    0.019608
NASA_B0005    0.000000
NASA_B0029    0.000000
Name: is_knee_candidate, dtype: float64

Cells where candidate rate is exactly 0 (never even momentarily crosses threshold): 25 / 33


/tmp/ipykernel_16/191337334.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_nasa_test = nasa_cells.groupby('cell_id', group_keys=False).apply(
/tmp/ipykernel_16/191337334.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_nasa_candidates = nasa_cells.groupby('cell_id', group_keys=False).apply(


In [29]:
# Cell 27: Inspect NASA_B0034's actual SoH trajectory around its candidate window --
# is this a real accelerating fade, or a single noisy dip that the rolling-window
# overlap turned into several consecutive "candidate" flags?

b34 = df_clean[df_clean['cell_id'] == 'NASA_B0034'].sort_values('cycle_index').reset_index(drop=True)
b34_detected = detect_knee_points_v3(b34, min_duration=5)

candidate_cycles = b34_detected[b34_detected['is_knee_candidate']]['cycle_index']
print("Candidate cycle range:", candidate_cycles.min(), "-", candidate_cycles.max())
print("\nSoH trajectory around and after the candidate window:")
window_start = max(candidate_cycles.min() - 10, 0) if len(candidate_cycles) else 0
print(b34_detected[['cycle_index','soh','rolling_fade_rate','is_knee_candidate']]
      .iloc[window_start: window_start + 40].to_string(index=False))

Candidate cycle range: 37 - 197

SoH trajectory around and after the candidate window:
 cycle_index      soh  rolling_fade_rate  is_knee_candidate
          29 0.736656           0.000420              False
          30 0.720797          -0.000008              False
          31 0.711294          -0.000363              False
          32 0.713324          -0.000481              False
          33 0.729839           0.000689              False
          34 0.719808           0.000322              False
          35 0.709776          -0.000088              False
          36 0.705905          -0.000728              False
          37 0.702210          -0.002062               True
          38 0.736753          -0.001276              False
          39 0.746329           0.000744              False
          40 0.741013           0.001474              False
          41 0.711788           0.000462              False
          42 0.701913          -0.005728               True
          43 

In [30]:
# Cell 28: Finalize ground truth (94 cells, CALCE/MIT/SNL), save labels

cell_knee_labels = df_knee3.groupby('cell_id')['is_knee_point'].any().reset_index()
cell_knee_labels.columns = ['cell_id', 'has_knee_point']

cell_info = df_knee3.groupby('cell_id').agg(
    source=('source','first'), cathode=('cathode','first'), 
    total_cycles=('cycle_index','max')
).reset_index()
cell_knee_labels = cell_knee_labels.merge(cell_info, on='cell_id')

# explicitly exclude NASA from the ground-truth set given the confirmed noise issue
cell_knee_labels_final = cell_knee_labels[cell_knee_labels['source'] != 'nasa'].copy()

print("Final ground truth (NASA excluded):")
print("Class balance:")
print(cell_knee_labels_final['has_knee_point'].value_counts())
print("\nBy source:")
print(cell_knee_labels_final.groupby(['source','has_knee_point']).size())

cell_knee_labels_final.to_csv('/kaggle/working/knee_point_labels.csv', index=False)
print("\nSaved knee_point_labels.csv —", len(cell_knee_labels_final), "cells")

Final ground truth (NASA excluded):
Class balance:
has_knee_point
False    115
True      94
Name: count, dtype: int64

By source:
source  has_knee_point
calce   False              1
        True               7
mit     False             65
        True              75
snl     False             49
        True              12
dtype: int64

Saved knee_point_labels.csv — 209 cells


In [31]:
# Cell 28: Finalize ground truth (94 cells, CALCE/MIT/SNL), save labels

cell_knee_labels = df_knee3.groupby('cell_id')['is_knee_point'].any().reset_index()
cell_knee_labels.columns = ['cell_id', 'has_knee_point']

cell_info = df_knee3.groupby('cell_id').agg(
    source=('source','first'), cathode=('cathode','first'), 
    total_cycles=('cycle_index','max')
).reset_index()
cell_knee_labels = cell_knee_labels.merge(cell_info, on='cell_id')

# explicitly exclude NASA from the ground-truth set given the confirmed noise issue
cell_knee_labels_final = cell_knee_labels[cell_knee_labels['source'] != 'nasa'].copy()

print("Final ground truth (NASA excluded):")
print("Class balance:")
print(cell_knee_labels_final['has_knee_point'].value_counts())
print("\nBy source:")
print(cell_knee_labels_final.groupby(['source','has_knee_point']).size())

cell_knee_labels_final.to_csv('/kaggle/working/knee_point_labels.csv', index=False)
print("\nSaved knee_point_labels.csv —", len(cell_knee_labels_final), "cells")

Final ground truth (NASA excluded):
Class balance:
has_knee_point
False    115
True      94
Name: count, dtype: int64

By source:
source  has_knee_point
calce   False              1
        True               7
mit     False             65
        True              75
snl     False             49
        True              12
dtype: int64

Saved knee_point_labels.csv — 209 cells


In [32]:
# Cell 29: Engineer early-life features per cell (reuse Stage 1/2 style), build the classification dataset

def engineer_knee_features(cell_id, df_source, early_pct=0.20):
    g = df_source[df_source['cell_id'] == cell_id].sort_values('cycle_index')
    g = g.dropna(subset=['soh'])
    if len(g) < 10:
        return None
    
    n_early = max(int(len(g) * early_pct), 10)
    early = g.iloc[:n_early]
    
    cycles = early['cycle_index'].values
    soh_vals = early['soh'].values
    
    slope, _ = np.polyfit(cycles, soh_vals, 1)
    curvature = np.polyfit(cycles, soh_vals, 2)[0] if len(early) >= 5 else np.nan
    
    return {
        'cell_id': cell_id,
        'soh_start': soh_vals[0],
        'soh_at_early_cutoff': soh_vals[-1],
        'early_fade_slope': slope,
        'early_fade_curvature': curvature,
        'early_soh_std': np.std(soh_vals),
        'n_early_cycles': n_early,
    }

feature_rows_knee = []
for cid in cell_knee_labels_final['cell_id']:
    feat = engineer_knee_features(cid, df_clean)
    if feat is not None:
        feature_rows_knee.append(feat)

knee_features_df = pd.DataFrame(feature_rows_knee)
knee_model_df = knee_features_df.merge(cell_knee_labels_final[['cell_id','source','cathode','has_knee_point']], on='cell_id', how='inner')

print("Shape:", knee_model_df.shape)
print("Null counts:")
print(knee_model_df.isna().sum())
print("\nClass balance in final model dataset:")
print(knee_model_df['has_knee_point'].value_counts())
print("\nSample:")
print(knee_model_df.head(10).to_string())

Shape: (209, 10)
Null counts:
cell_id                 0
soh_start               0
soh_at_early_cutoff     0
early_fade_slope        0
early_fade_curvature    0
early_soh_std           0
n_early_cycles          0
source                  0
cathode                 0
has_knee_point          0
dtype: int64

Class balance in final model dataset:
has_knee_point
False    115
True      94
Name: count, dtype: int64

Sample:
                cell_id  soh_start  soh_at_early_cutoff  early_fade_slope  early_fade_curvature  early_soh_std  n_early_cycles source cathode  has_knee_point
0                CS2_21   0.881818             0.845455         -0.000265         -1.032926e-06       0.017581             156  calce     LCO            True
1                CS2_33   0.950941             0.896711         -0.000381          1.661621e-06       0.023042             109  calce     LCO            True
2                CS2_34   0.956204             0.921076         -0.000219          1.378460e-06       0.0188

In [33]:
# Cell 30: Train RandomForestClassifier, StratifiedKFold evaluation

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

feature_cols_knee = ['soh_start', 'soh_at_early_cutoff', 'early_fade_slope', 
                      'early_fade_curvature', 'early_soh_std', 'n_early_cycles']

X_knee = knee_model_df[feature_cols_knee].fillna(-1)
y_knee = knee_model_df['has_knee_point'].astype(int)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_preds_knee = np.zeros(len(knee_model_df))
all_probs_knee = np.zeros(len(knee_model_df))

for fold, (train_idx, test_idx) in enumerate(skf.split(X_knee, y_knee)):
    clf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, min_samples_leaf=3)
    clf.fit(X_knee.iloc[train_idx], y_knee.iloc[train_idx])
    all_preds_knee[test_idx] = clf.predict(X_knee.iloc[test_idx])
    all_probs_knee[test_idx] = clf.predict_proba(X_knee.iloc[test_idx])[:, 1]

acc = accuracy_score(y_knee, all_preds_knee)
prec = precision_score(y_knee, all_preds_knee)
rec = recall_score(y_knee, all_preds_knee)
f1 = f1_score(y_knee, all_preds_knee)
auc = roc_auc_score(y_knee, all_probs_knee)

print(f"Accuracy: {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall: {rec:.3f}")
print(f"F1: {f1:.3f}")
print(f"ROC-AUC: {auc:.3f}")

knee_cv_results = knee_model_df[['cell_id','source','has_knee_point']].copy()
knee_cv_results['predicted'] = all_preds_knee
knee_cv_results['predicted_prob'] = all_probs_knee

print("\nPer-source accuracy:")
print(knee_cv_results.groupby('source').apply(lambda g: accuracy_score(g['has_knee_point'], g['predicted'])))

print("\nBaseline (majority class) accuracy for comparison:", max(y_knee.mean(), 1-y_knee.mean()))

Accuracy: 0.837
Precision: 0.841
Recall: 0.787
F1: 0.813
ROC-AUC: 0.885

Per-source accuracy:
source
calce    0.875000
mit      0.835714
snl      0.836066
dtype: float64

Baseline (majority class) accuracy for comparison: 0.5502392344497608


/tmp/ipykernel_16/85371211.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(knee_cv_results.groupby('source').apply(lambda g: accuracy_score(g['has_knee_point'], g['predicted'])))


In [34]:
# Cell 31: Save final model, results, and Stage 3 summary
import joblib
final_knee_clf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, min_samples_leaf=3)
final_knee_clf.fit(X_knee, y_knee)
joblib.dump(final_knee_clf, '/kaggle/working/knee_point_final_model.pkl')

knee_cv_results.to_csv('/kaggle/working/knee_point_cv_results.csv', index=False)

importances_knee = pd.Series(final_knee_clf.feature_importances_, index=feature_cols_knee).sort_values(ascending=False)
print("Feature importances:")
print(importances_knee)

stage3_summary = {
    'model': 'RandomForestClassifier',
    'target': 'has_knee_point (binary: sustained accelerating degradation detected)',
    'validation_method': 'StratifiedKFold (5-fold)',
    'cv_accuracy': acc,
    'cv_precision': prec,
    'cv_recall': rec,
    'cv_f1': f1,
    'cv_roc_auc': auc,
    'naive_baseline_accuracy': max(y_knee.mean(), 1-y_knee.mean()),
    'n_cells_total': len(knee_model_df),
    'per_source_accuracy': knee_cv_results.groupby('source').apply(lambda g: accuracy_score(g['has_knee_point'], g['predicted'])).to_dict(),
    'excluded_source': 'nasa (33 cells) -- confirmed via sensitivity analysis to have inherently noisy '
                        'cycle-to-cycle SOH data that prevents reliable sustained-trend detection, not '
                        'a detector threshold issue. Documented as a known data-quality limitation.',
    'ground_truth_method': 'Sustained rolling fade-rate acceleration (20-cycle window, min 10 consecutive '
                            'cycles beyond 2.5x baseline rate and absolute minimum 0.001 SOH/cycle decline) '
                            '-- physically grounded in the well-documented battery "knee point" phenomenon, '
                            'not derived from data-cleaning artifacts.',
}

import json
with open('/kaggle/working/stage3_summary.json', 'w') as f:
    json.dump(stage3_summary, f, indent=2, default=str)

print("\n=== STAGE 3 SUMMARY ===")
for k, v in stage3_summary.items():
    print(f"  {k}: {v}")

Feature importances:
early_fade_curvature    0.254616
n_early_cycles          0.236603
soh_start               0.160760
early_fade_slope        0.135545
early_soh_std           0.109240
soh_at_early_cutoff     0.103236
dtype: float64

=== STAGE 3 SUMMARY ===
  model: RandomForestClassifier
  target: has_knee_point (binary: sustained accelerating degradation detected)
  validation_method: StratifiedKFold (5-fold)
  cv_accuracy: 0.8373205741626795
  cv_precision: 0.8409090909090909
  cv_recall: 0.7872340425531915
  cv_f1: 0.8131868131868132
  cv_roc_auc: 0.8849213691026828
  naive_baseline_accuracy: 0.5502392344497608
  n_cells_total: 209
  per_source_accuracy: {'calce': 0.875, 'mit': 0.8357142857142857, 'snl': 0.8360655737704918}
  excluded_source: nasa (33 cells) -- confirmed via sensitivity analysis to have inherently noisy cycle-to-cycle SOH data that prevents reliable sustained-trend detection, not a detector threshold issue. Documented as a known data-quality limitation.
  ground_t

/tmp/ipykernel_16/2998406531.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  'per_source_accuracy': knee_cv_results.groupby('source').apply(lambda g: accuracy_score(g['has_knee_point'], g['predicted'])).to_dict(),
